In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
listings = pd.read_csv("../Data/raw/Listings.csv", encoding="latin-1")
reviews = pd.read_csv("../Data/raw/Reviews.csv", encoding="latin-1")

C:\Users\Parv Gupta\AppData\Local\Temp\ipykernel_15128\2050404675.py:1: DtypeWarning: Columns (5,13) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("../data/raw/Listings.csv", encoding="latin-1")


In [3]:
listings_clean = listings.copy()
reviews_clean = reviews.copy()

In [4]:
print("Original Shape:", listings.shape)
print("Working Copy Shape:", listings_clean.shape)

Original Shape: (279712, 33)
Working Copy Shape: (279712, 33)


### Decision: district

Problem:
- Approximately 87% missing values.

Business Evaluation:
- Geographic information is already available through:
  - city
  - neighbourhood
  - latitude
  - longitude

Decision:
- Drop the `district` column during data cleaning.

Reason:
- High missingness combined with low additional business value.

In [5]:
listings_clean = listings_clean.drop(columns=["district"])
listings_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 32 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   listing_id                   279712 non-null  int64  
 1   name                         279537 non-null  object 
 2   host_id                      279712 non-null  int64  
 3   host_since                   279547 non-null  object 
 4   host_location                278872 non-null  object 
 5   host_response_time           150930 non-null  object 
 6   host_response_rate           150930 non-null  float64
 7   host_acceptance_rate         166625 non-null  float64
 8   host_is_superhost            279547 non-null  object 
 9   host_total_listings_count    279547 non-null  float64
 10  host_has_profile_pic         279547 non-null  object 
 11  host_identity_verified       279547 non-null  object 
 12  neighbourhood                279712 non-null  object 
 13 

In [6]:
missing_hosts=listings_clean[
    ["host_response_time",
     "host_response_rate",
     "host_acceptance_rate"]
].isna().all(axis=1)

missing_hosts.describe()

count     279712
unique         2
top        False
freq      182543
dtype: object

In [7]:
listings_clean["host_response_time"].isna().sum()

np.int64(128782)

In [8]:
listings_clean["host_response_rate"].isna().sum()

np.int64(128782)

In [9]:
listings_clean["host_acceptance_rate"].isna().sum()

np.int64(113087)

In [10]:
listings_clean.columns.tolist()

['listing_id',
 'name',
 'host_id',
 'host_since',
 'host_location',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_total_listings_count',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'city',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bedrooms',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value',
 'instant_bookable']

In [11]:
missing_hosts = listings_clean[
    listings_clean[
        ["host_response_time",
         "host_response_rate",
         "host_acceptance_rate"]
    ].isna().all(axis=1)
]

print(missing_hosts["review_scores_rating"].isna().sum())
print(len(missing_hosts))

45506
97169


## Cleaning Decision – Host Response Metrics

### Observation
- `host_response_time`, `host_response_rate`, and `host_acceptance_rate` contain a substantial number of missing values.
- 97,169 listings are missing all three metrics simultaneously.
- Over 50,000 of these listings still have review scores, indicating that the missing values are not simply due to a lack of guest reviews.

### Decision
Retain all three columns and preserve the missing values.

### Reason
The missing values appear to follow an unknown business rule rather than representing random data quality issues. Imputing or deleting these values at this stage could distort future analyses.

In [12]:
zero_price = listings_clean[listings_clean["price"] == 0]
zero_price.to_csv("../data/zero_price_listings.csv", index=False)

In [13]:
listings_clean.loc[
    listings_clean["price"] == 0,
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "minimum_nights",
        "maximum_nights",
        "review_scores_rating"
    ]
]

,listing_id,city,property_type,room_type,minimum_nights,maximum_nights,review_scores_rating
98209,44312919,Paris,Room in boutique hotel,Hotel room,1,1125,NaN
202022,42830099,New York,Room in boutique hotel,Hotel room,30,365,NaN
202023,43247472,New York,Room in boutique hotel,Hotel room,30,365,NaN
203253,43205598,New York,Room in boutique hotel,Hotel room,1,365,NaN
203254,44567521,New York,Room in boutique hotel,Hotel room,1,365,NaN
...,...,...,...,...,...,...,...
208940,42534857,Rome,Room in hotel,Hotel room,1,365,NaN
208941,43036135,Rome,Room in boutique hotel,Hotel room,1,365,NaN
210617,42065564,New York,Room in hotel,Hotel room,1,30,NaN
212834,43308777,Paris,Room in hostel,Hotel room,1,365,NaN


In [14]:
listings_clean[
    (listings_clean["price"] == 0) &
    (listings_clean["accommodates"] == 0)
].shape

(85, 32)

In [15]:
listings_clean.loc[
    (listings_clean["price"] == 0) &
    (listings_clean["accommodates"] > 0),
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "accommodates",
        "bedrooms",
        "price",
        "minimum_nights",
        "maximum_nights",
        "review_scores_rating"
    ]
]

,listing_id,city,property_type,room_type,accommodates,bedrooms,price,minimum_nights,maximum_nights,review_scores_rating
203547,42431489,New York,Room in hotel,Hotel room,4,NaN,0,1,365,NaN
203605,41740615,New York,Room in boutique hotel,Hotel room,4,NaN,0,1,28,NaN
203606,41792753,New York,Room in boutique hotel,Hotel room,6,NaN,0,1,365,NaN
203607,42065562,New York,Room in hotel,Hotel room,4,NaN,0,30,365,NaN
203608,42279171,New York,Room in boutique hotel,Hotel room,6,NaN,0,1,365,NaN
203609,42384501,New York,Room in boutique hotel,Hotel room,2,NaN,0,1,28,NaN
206709,42065547,New York,Room in hotel,Hotel room,4,NaN,0,30,365,NaN
206710,42065555,New York,Room in hotel,Hotel room,2,NaN,0,30,365,NaN
206711,42279170,New York,Room in boutique hotel,Hotel room,2,NaN,0,30,365,NaN
206712,42384530,New York,Room in boutique hotel,Hotel room,2,NaN,0,30,365,NaN


In [16]:
listings_clean[listings_clean["price"] == 0]["review_scores_rating"].isna().sum()

np.int64(113)

In [17]:
listings_clean[listings_clean["price"] == 0].shape[0]

113

In [18]:
listings_clean = listings_clean[listings_clean["price"] > 0]

In [19]:
(listings_clean["accommodates"] == 0).sum()

np.int64(0)

In [20]:
(listings_clean["maximum_nights"] == 2147483647).sum()

np.int64(3)

In [21]:
listings_clean.loc[
    listings_clean["maximum_nights"] == 2147483647,
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "price",
        "minimum_nights",
        "maximum_nights"
    ]
]

,listing_id,city,property_type,room_type,price,minimum_nights,maximum_nights
228967,4234075,Rome,Entire apartment,Entire place,100,2,2147483647
251162,6357527,New York,Entire apartment,Entire place,130,30,2147483647
259338,744242,Istanbul,Entire apartment,Entire place,289,2,2147483647


In [22]:
listings_clean["maximum_nights"] = listings_clean["maximum_nights"].replace(
    2147483647,
    np.nan
)

In [23]:
(listings_clean["maximum_nights"] == 2147483647).sum()

np.int64(0)

## Cleaning Step 3 - Handle Placeholder Values in `maximum_nights`

### Problem
Three listings had a value of `2147483647` for `maximum_nights`.

### Investigation
The listings appeared valid in every other aspect (reasonable prices, property types, and minimum nights). The value `2147483647` is a common system placeholder rather than a realistic maximum stay.

### Decision
Replaced `2147483647` with `NaN`.

### Reason
The placeholder does not represent a meaningful numeric value. Using `NaN` prevents it from distorting statistical analysis while indicating that the actual maximum is unknown or not explicitly defined.

In [24]:
listings_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 279599 entries, 0 to 279711
Data columns (total 32 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   listing_id                   279599 non-null  int64  
 1   name                         279424 non-null  object 
 2   host_id                      279599 non-null  int64  
 3   host_since                   279434 non-null  object 
 4   host_location                278759 non-null  object 
 5   host_response_time           150898 non-null  object 
 6   host_response_rate           150898 non-null  float64
 7   host_acceptance_rate         166577 non-null  float64
 8   host_is_superhost            279434 non-null  object 
 9   host_total_listings_count    279434 non-null  float64
 10  host_has_profile_pic         279434 non-null  object 
 11  host_identity_verified       279434 non-null  object 
 12  neighbourhood                279599 non-null  object 
 13  city

In [25]:
(
    listings_clean["bedrooms"].isna().sum() /
    len(listings_clean)
) * 100

np.float64(10.487161971251686)

In [26]:
listings_clean[listings_clean["bedrooms"].isna()]["property_type"].value_counts().head(15)

property_type
Entire apartment                      20527
Private room in apartment              1868
Entire condominium                     1278
Entire loft                             815
Private room in house                   496
Entire serviced apartment               492
Room in boutique hotel                  429
Entire guest suite                      401
Entire house                            342
Private room in condominium             333
Room in hotel                           281
Room in aparthotel                      259
Private room in bed and breakfast       224
Entire guesthouse                       160
Private room in serviced apartment      151
Name: count, dtype: int64

In [27]:
listings_clean[listings_clean["bedrooms"].isna()]["room_type"].value_counts()

room_type
Entire place    24569
Private room     4398
Hotel room        355
Name: count, dtype: int64

In [28]:
listings_clean["bedrooms"].describe()

count    250277.000000
mean          1.515509
std           1.153080
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          50.000000
Name: bedrooms, dtype: float64

## Cleaning Step 4 - Bedrooms

### Problem
29,322 listings (10.49%) have missing bedroom values.

### Investigation
- Missing values occur across multiple property types, especially entire apartments.
- Median bedroom count is 1, but applying it to all missing values would create unrealistic records for larger properties.

### Decision
Retain the missing values.

### Reason
A simple imputation is likely to introduce bias. The missing values will be handled later if required during feature engineering or model development.

In [29]:
listings_clean[
    [
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value",
    ]
].isna().sum()

review_scores_rating           91292
review_scores_accuracy         91600
review_scores_cleanliness      91552
review_scores_checkin          91658
review_scores_communication    91574
review_scores_location         91662
review_scores_value            91672
dtype: int64

In [30]:
listings_clean[
    [
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value"
    ]
].isna().all(axis=1).sum()

np.int64(91264)

## Cleaning Step 5 - Review Score Columns

### Problem
The review score columns contain approximately 91,000 missing values.

### Investigation
The missing values occur together across almost all review score columns, indicating a common cause rather than random missingness.

### Decision
Retain the columns and preserve the missing values.

### Reason
The missing values most likely indicate listings without sufficient guest reviews rather than poor data quality. Imputing values would introduce misleading information.

In [31]:
missing = listings_clean.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

host_response_time             128701
host_response_rate             128701
host_acceptance_rate           113022
review_scores_value             91672
review_scores_location          91662
review_scores_checkin           91658
review_scores_accuracy          91600
review_scores_communication     91574
review_scores_cleanliness       91552
review_scores_rating            91292
bedrooms                        29322
host_location                     840
name                              175
host_has_profile_pic              165
host_identity_verified            165
host_is_superhost                 165
host_since                        165
host_total_listings_count         165
maximum_nights                      3
dtype: int64

In [32]:
missing_df = pd.DataFrame({
    "Missing Count": listings_clean.isna().sum(),
    "Missing %": (listings_clean.isna().mean() * 100).round(2)
})

missing_df = missing_df[missing_df["Missing Count"] > 0]
missing_df.sort_values("Missing %", ascending=False)

,Missing Count,Missing %
host_response_time,128701,46.03
host_response_rate,128701,46.03
host_acceptance_rate,113022,40.42
review_scores_value,91672,32.79
review_scores_location,91662,32.78
review_scores_checkin,91658,32.78
review_scores_accuracy,91600,32.76
review_scores_communication,91574,32.75
review_scores_cleanliness,91552,32.74
review_scores_rating,91292,32.65


In [33]:
listings_clean.duplicated().sum()

np.int64(0)

In [34]:
listings_clean["listing_id"].duplicated().sum()

np.int64(0)

In [35]:
listings_clean.dtypes

listing_id                       int64
name                            object
host_id                          int64
host_since                      object
host_location                   object
host_response_time              object
host_response_rate             float64
host_acceptance_rate           float64
host_is_superhost               object
host_total_listings_count      float64
host_has_profile_pic            object
host_identity_verified          object
neighbourhood                   object
city                            object
latitude                       float64
longitude                      float64
property_type                   object
room_type                       object
accommodates                     int64
bedrooms                       float64
amenities                       object
price                            int64
minimum_nights                   int64
maximum_nights                 float64
review_scores_rating           float64
review_scores_accuracy   

In [36]:
listings_clean["host_since"] = pd.to_datetime(
    listings_clean["host_since"],
    errors="coerce"
)

In [37]:
listings_clean["host_since"].dtype

dtype('<M8[ns]')

In [38]:
bool_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable"
]

for col in bool_cols:
    listings_clean[col] = listings_clean[col].map({"t": True, "f": False})

In [39]:
listings_clean[bool_cols].dtypes

host_is_superhost         object
host_has_profile_pic      object
host_identity_verified    object
instant_bookable            bool
dtype: object

In [40]:
bool_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable"
]

for col in bool_cols:
    listings_clean[col] = listings_clean[col].replace({
        "t": True,
        "f": False
    })

In [41]:
listings_clean[bool_cols].dtypes

host_is_superhost         object
host_has_profile_pic      object
host_identity_verified    object
instant_bookable            bool
dtype: object

In [42]:
listings_clean.to_csv("../data/cleaned/Listings_Clean.csv", index=False)

reviews_clean.to_csv("../data/cleaned/Reviews_Clean.csv", index=False)

In [43]:
listings_clean["host_is_superhost"].value_counts(dropna=False)

host_is_superhost
False    229185
True      50249
NaN         165
Name: count, dtype: int64

In [44]:
listings["host_is_superhost"].head()

0    f
1    f
2    f
3    f
4    f
Name: host_is_superhost, dtype: object

In [45]:
listings_clean["host_is_superhost"].value_counts(dropna=False)

host_is_superhost
False    229185
True      50249
NaN         165
Name: count, dtype: int64

In [46]:
listings_clean["host_is_superhost"].head(10)


0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: host_is_superhost, dtype: object

In [47]:
listings_clean["host_is_superhost"].unique()

array([False, True, nan], dtype=object)

In [48]:
listings_clean["host_is_superhost"].dtype

dtype('O')

In [49]:
listings_clean.to_csv("../data/cleaned/Listings_Clean.csv", index=False)

reviews_clean.to_csv("../data/cleaned/Reviews_Clean.csv", index=False)